# Lab 09: Sectra First Responder, Incident-Response Chatbot (Jupyter)

**Course:** AI for Cybersecurity and Digital Forensics  ·  **SectraTEK Lab**

The same incident-response copilot, run **in the notebook**: call `talk("...")` and read Sectra's
steps and the playbook section it used. Sectra remembers the conversation, so approvals work across
calls. This is a **drill**: emails are simulated. A sample plan + playbooks and two sample artifacts
are bundled; drop your own playbooks into `knowledge/` and artifacts into `evidence/` to use real ones.

### 1. Install the client, create folders, write the code + sample playbooks/evidence

In [ ]:
%pip install -q openai pypdf python-docx

In [ ]:
import os
os.makedirs('knowledge', exist_ok=True)
os.makedirs('evidence', exist_ok=True)

In [ ]:
%%writefile agentkit.py
"""
agentkit.py - the shared forensic-agent core used by the Lab 07-10 notebooks and chatbots.

One place defines: how to connect to a model (Groq, OpenRouter or Ollama), how to build the
synthetic Android image, the read-only tools the agent may call, the think-act-observe
loop with its evidence check, and a conversational wrapper for the chatbots. The chatbot .py apps import this
module; the teaching notebooks embed the same code inline so students open only one file.
"""
import contextlib
import inspect
import json
import os
import re
import shutil
import sqlite3
import tempfile
from pathlib import Path


def as_text(content):
    """Message content as plain text. Gradio 6 can hand a chat message back as a list of parts
    ([{'type': 'text', 'text': ...}]) instead of a string; every caller goes through this."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join((x.get("text") or x.get("content") or "") if isinstance(x, dict) else str(x)
                       for x in content)
    if isinstance(content, dict):
        return str(content.get("text") or content.get("content") or "")
    return str(content or "")


@contextlib.contextmanager
def sqlite_copy(path):
    """Open a SQLite evidence file WITHOUT touching it: query a temporary copy (with its -wal and
    -journal files, so recent rows are included). Opening the original, even read-only, can create
    -shm/-wal files beside it, and a path with '#' or '?' can be misread as a different file."""
    path = Path(path)
    tmp = Path(tempfile.mkdtemp(prefix="sectra_db_"))
    try:
        shutil.copy2(path, tmp / "evidence.db")
        for side in ("-wal", "-journal"):
            src = path.with_name(path.name + side)
            if src.exists():
                shutil.copy2(src, tmp / ("evidence.db" + side))
        con = sqlite3.connect(tmp / "evidence.db")
        try:
            yield con
        finally:
            con.close()
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def read_head(path, n=16):
    """The first n bytes of a file (its 'magic'), without reading the whole file into memory."""
    with open(path, "rb") as f:
        return f.read(n)


# ======================================================================= model backend
# Free models that follow the agent's JSON format, best first. Providers retire models from
# time to time, so connect_model picks the first one your key can actually use, and the
# notebooks print which one it chose. Put your own choice first, or pass model="...".
GROQ_MODELS = ["openai/gpt-oss-120b", "qwen/qwen3.8-27b"]
OPENROUTER_MODELS = ["google/gemma-4-31b-it:free", "nvidia/nemotron-3-ultra-550b-a55b:free"]
NOT_CHAT = ("whisper", "orpheus", "guard", "tts", "embed")   # speech / safety / embedding models


def pick_model(client, preferred, free_only=False):
    """Return the first preferred model this key can use, else the largest chat model it lists."""
    try:
        listed = client.models.list().data
    except Exception:
        return preferred[0]          # cannot list models: try the first choice and let it report
    ids = [m.id for m in listed]
    for m in preferred:
        if m in ids:
            return m
    chat = [m for m in listed if not any(w in m.id.lower() for w in NOT_CHAT)
            and (m.id.endswith(":free") or not free_only)]
    chat.sort(key=lambda m: getattr(m, "context_window", 0) or 0, reverse=True)
    return chat[0].id if chat else preferred[0]


def connect_model(provider="groq", model=None, api_key=None, base_url=None):
    """Return (client, model_id). provider is 'groq' (default), 'openrouter' or 'ollama'.

    groq:       free hosted models, key from https://console.groq.com/keys (the same key as Labs 01-06).
    openrouter: free hosted models, key from https://openrouter.ai/keys.
    ollama:     a Qwen model running locally (https://ollama.com), no key, nothing leaves the PC.

    max_retries=8: the free tiers allow a few thousand tokens per minute. When a step goes over,
    the provider says "try again in N seconds" and the client waits and retries instead of failing.
    """
    from openai import OpenAI
    provider = provider.lower()
    if provider == "ollama":
        base_url = base_url or "http://localhost:11434/v1"
        client = OpenAI(base_url=base_url, api_key="ollama")  # Ollama ignores the key
        return client, (model or "qwen3:8b")
    if provider == "openrouter":
        client = OpenAI(base_url=base_url or "https://openrouter.ai/api/v1",
                        api_key=api_key or os.environ.get("OPENROUTER_API_KEY", ""), max_retries=8)
        return client, (model or pick_model(client, OPENROUTER_MODELS, free_only=True))
    # default: Groq
    client = OpenAI(base_url=base_url or "https://api.groq.com/openai/v1",
                    api_key=api_key or os.environ.get("GROQ_API_KEY", ""), max_retries=8)
    return client, (model or pick_model(client, GROQ_MODELS))


# ======================================================================= synthetic image
def build_image(root="android_image"):
    """Create a tiny, synthetic Android extraction on disk. Everything in it is fictional."""
    ROOT = Path(root)
    text_files = {
        f"{root}/build.prop":
            "# Synthetic build.prop (teaching data, fictional)\n"
            "ro.product.manufacturer=Google\nro.product.brand=google\n"
            "ro.product.model=Pixel 7a\nro.product.device=lynx\nro.product.name=lynx\n"
            "ro.build.version.release=14\nro.build.version.sdk=34\n"
            "ro.build.fingerprint=google/lynx/lynx:14/AP2A.240705.005/2024072800:user/release-keys\n",
        f"{root}/data/system/packages.list":
            "com.google.android.gm 10089 0 /data/user/0/com.google.android.gm default 0 1\n"
            "com.android.providers.contacts 10003 0 /data/user/0/com.android.providers.contacts platform 0 1\n"
            "com.android.providers.telephony 10004 0 /data/user/0/com.android.providers.telephony platform 0 1\n"
            "com.whatsapp 10137 0 /data/user/0/com.whatsapp default 0 1\n"
            "com.instagram.android 10140 0 /data/user/0/com.instagram.android default 0 1\n"
            "org.thoughtcrime.securesms 10142 0 /data/user/0/org.thoughtcrime.securesms default 0 1\n",
        f"{root}/data/system/users/0/settings_secure.xml":
            "<?xml version='1.0' encoding='utf-8' standalone='yes' ?>\n<settings version=\"241\">\n"
            "  <setting id=\"1\" name=\"android_id\" value=\"a1b2c3d4e5f60718\" package=\"android\" />\n"
            "  <setting id=\"2\" name=\"bluetooth_name\" value=\"Pixel 7a\" package=\"com.android.settings\" />\n"
            "  <setting id=\"3\" name=\"device_name\" value=\"Nadia's Pixel 7a\" package=\"com.android.settings\" />\n"
            "</settings>\n",
        f"{root}/data/misc/wifi/WifiConfigStore.xml":
            "<?xml version='1.0' encoding='utf-8' standalone='yes' ?>\n<WifiConfigStoreData>\n  <NetworkList>\n"
            "    <Network><string name=\"SSID\">&quot;HomeNet_5G&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1721295600000\" /></Network>\n"
            "    <Network><string name=\"SSID\">&quot;BrightBeans Cafe&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1720950000000\" /></Network>\n"
            "    <Network><string name=\"SSID\">&quot;Okafor-Office&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1721210400000\" /></Network>\n"
            "  </NetworkList>\n</WifiConfigStoreData>\n",
        # a JSON artifact (many Android apps store prefs/state as JSON)
        f"{root}/data/data/com.whatsapp/files/me.json":
            json.dumps({"jid": "15550143788@s.whatsapp.net", "pushname": "Nadia O",
                        "registered": True, "last_backup": "2024-07-18"}, indent=2),
    }
    for rel, content in text_files.items():
        p = Path(rel)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content)

    def make_db(rel, schema, inserts):
        p = Path(rel)
        p.parent.mkdir(parents=True, exist_ok=True)
        if p.exists():
            p.unlink()
        con = sqlite3.connect(p)
        con.executescript(schema)
        for sql, params in inserts:
            con.executemany(sql, params)
        con.commit()
        con.close()

    make_db(f"{root}/data/data/com.google.android.gm/databases/accounts.db",
            "CREATE TABLE accounts(name TEXT, type TEXT, last_sync INTEGER);",
            [("INSERT INTO accounts VALUES (?,?,?)",
              [("nadia.okafor@gmail.com", "com.google", 1721295600),
               ("+15550143788", "com.whatsapp", 1721240000)])])
    make_db(f"{root}/data/data/com.android.providers.contacts/databases/contacts2.db",
            "CREATE TABLE contacts(display_name TEXT, phone TEXT, times_contacted INTEGER);",
            [("INSERT INTO contacts VALUES (?,?,?)",
              [("Marcus Webb", "+15550142210", 31), ("Priya Shah", "+15550148890", 12),
               ("Dad", "+15550149001", 54), ("BrightBeans Cafe", "+15550140077", 3),
               ("Alina K (work)", "+15550147325", 20)])])
    make_db(f"{root}/data/data/com.android.providers.telephony/databases/mmssms.db",
            "CREATE TABLE sms(address TEXT, date INTEGER, body TEXT, type INTEGER);",
            [("INSERT INTO sms VALUES (?,?,?,?)",
              [("+15550142210", 1721200000000, "Are we still on for Friday?", 1),
               ("+15550142210", 1721200600000, "Yes, 6pm at the usual place.", 2),
               ("+15550149001", 1721240500000, "Landed safe, call you tonight.", 2),
               ("+15550148890", 1721250000000, "Can you send the invoice?", 1),
               ("+15550149001", 1721299000000, "Thanks Dad, talk soon.", 2)])])
    make_db(f"{root}/data/data/com.whatsapp/databases/msgstore.db",
            "CREATE TABLE messages(chat TEXT, timestamp INTEGER, from_me INTEGER, text TEXT);",
            [("INSERT INTO messages VALUES (?,?,?,?)",
              [("Marcus Webb", 1721260000000, 0, "Did the package arrive?"),
               ("Marcus Webb", 1721260400000, 1, "Not yet, will check tomorrow."),
               ("Priya Shah", 1721280000000, 1, "Sent the files to your email.")])])
    return ROOT


# ======================================================================= the tools
# A text-ish file is one the agent can read as text. Everything else is "binary".
_TEXT_EXT = {".txt", ".xml", ".json", ".prop", ".list", ".log", ".csv", ".ini", ".conf", ".html"}


def make_tools(root="android_image", allow_python=False):
    """Build the read-only toolbox, all sealed inside `root`. Returns {name: function}."""
    ROOT = Path(root)

    def _resolve(path):
        base = ROOT.resolve()
        path = (path or ".").strip().lstrip("/")
        target = base if path in ("", ".") else (ROOT / path).resolve()
        if target != base and base not in target.parents:
            raise ValueError("path is outside the image")
        return target

    def list_dir(path="."):
        """List what is inside a folder (folders shown with a trailing '/')."""
        t = _resolve(path)
        if not t.exists():
            return f"No such path: {path}"
        if t.is_file():
            return f"{path} is a file, not a folder. Use read_file or run_sql."
        items = [e + ("/" if (t / e).is_dir() else "") for e in sorted(os.listdir(t))]
        return "\n".join(items) if items else "(empty folder)"

    def read_file(path, max_chars=1500):
        """Read a file. Pretty-prints JSON; for SQLite databases points you to run_sql;
        for other binary files shows the type and the first bytes."""
        t = _resolve(path)
        if not t.exists() or t.is_dir():
            return f"No such file: {path}"
        head = read_head(t)
        if head[:15] == b"SQLite format 3":
            with sqlite_copy(t) as con:
                tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
            return f"[{path} is a SQLite database. Tables: {tables}. Use run_sql with a SELECT query.]"
        ext = t.suffix.lower()
        if ext not in _TEXT_EXT and b"\x00" in head:
            return (f"[{path} is a binary file ({t.stat().st_size} bytes). First bytes (hex): "
                    f"{head.hex(' ')}. It is not readable as text.]")
        text = t.read_text(errors="replace")
        if ext == ".json":
            try:
                text = json.dumps(json.loads(text), indent=2)
            except Exception:
                pass
        return text[:max_chars] + ("" if len(text) <= max_chars else "\n...[truncated]")

    def run_sql(path, query):
        """Run ONE read-only SQL SELECT against a SQLite database and return the rows."""
        t = _resolve(path)
        if not t.exists():
            return f"No such database: {path}"
        q = (query or "").strip().rstrip(";")
        if not q.lower().startswith(("select", "pragma")):
            return "Only read-only SELECT queries are allowed."
        try:
            with sqlite_copy(t) as con:   # a temporary copy: the evidence file is never opened
                cur = con.execute(q)
                cols = [c[0] for c in cur.description] if cur.description else []
                rows = cur.fetchmany(50)
        except Exception as e:
            return f"SQL error: {e}"
        if not rows:
            return "(no rows)"
        return "\n".join([" | ".join(cols)] + [" | ".join(str(c) for c in r) for r in rows])

    def find_files(name):
        """Find files anywhere in the image whose path contains `name` (case-insensitive)."""
        base = ROOT.resolve()
        needle = (name or "").lower()
        hits = []
        for p in sorted(base.rglob("*")):
            if p.is_symlink():
                continue                      # never follow a link out of the image
            if p.is_file() and needle in str(p.relative_to(base)).lower():
                hits.append(str(p.relative_to(base)))
        if not hits:
            return f"No files matched {name!r}."
        return "\n".join(hits[:50]) + ("" if len(hits) <= 50 else f"\n...({len(hits)} total)")

    def search_text(keyword, max_hits=20):
        """Search the text inside readable files for `keyword` (case-insensitive).
        Returns path:line for each match, so you know which file to open next."""
        base = ROOT.resolve()
        needle = (keyword or "").lower()
        out = []
        for p in sorted(base.rglob("*")):
            if p.is_symlink() or not p.is_file() or p.suffix.lower() not in _TEXT_EXT:
                continue
            try:
                for i, line in enumerate(p.read_text(errors="replace").splitlines(), 1):
                    if needle in line.lower():
                        out.append(f"{p.relative_to(base)}:{i}: {line.strip()[:120]}")
                        if len(out) >= max_hits:
                            return "\n".join(out) + "\n...[more matches not shown]"
            except Exception:
                continue
        return "\n".join(out) if out else f"No text matches for {keyword!r}."

    tools = {"list_dir": list_dir, "read_file": read_file, "run_sql": run_sql,
             "find_files": find_files, "search_text": search_text}

    if allow_python:
        def run_python(code):
            """ADVANCED, OPTIONAL: run a short Python snippet to inspect the image. `ROOT` (the image
            folder, a pathlib.Path) is available. Print what you want to see. Do not write or delete."""
            # NOT a sandbox: the keyword check below stops accidents, not a determined snippet (Python
            # can still import modules and touch files). Use it only on a COPY of an extraction.
            import io
            import contextlib
            if any(bad in code for bad in ("open(", "write", "unlink", "remove", "rmtree",
                                           "os.system", "subprocess", "Popen", "__import__")):
                return "Blocked: that snippet looks like it writes or runs external commands."
            buf = io.StringIO()
            safe = {"ROOT": ROOT, "sqlite3": sqlite3, "json": json, "Path": Path, "re": re}
            try:
                with contextlib.redirect_stdout(buf):
                    exec(code, safe)  # teaching tool only; see the warning above
            except Exception as e:
                return f"Python error: {e}\n{buf.getvalue()}"
            out = buf.getvalue().strip()
            return out[:1500] if out else "(the snippet ran but printed nothing)"
        tools["run_python"] = run_python

    return tools


def tool_descriptions(tools):
    """One line per tool for the system prompt: its real signature (so the model uses the right
    argument names) and the first paragraph of its docstring."""
    lines = []
    for name, fn in tools.items():
        try:
            sig = str(inspect.signature(fn))
        except (TypeError, ValueError):
            sig = "()"
        doc = " ".join((fn.__doc__ or "").strip().split("\n\n")[0].split())
        lines.append(f"- {name}{sig}: {doc}")
    return "\n".join(lines)


# ======================================================================= the agent
def build_system_prompt(tools):
    return (
        "You are Sectra, a digital-forensics examiner's assistant. You are investigating an Android "
        "device extraction (a copy of the phone's files) to answer the examiner's question. You cannot "
        "see any file until you open it with a tool, and you must base every answer only on what the "
        "tools return.\n\n"
        "TOOLS (use exactly one per step):\n" + tool_descriptions(tools) + "\n\n"
        "HOW TO REPLY: send ONE JSON object and nothing else.\n"
        "To use a tool:\n"
        '  {"thought": "<one short sentence>", "action": "list_dir", "action_input": {"path": "data/data"}}\n'
        "When you can answer, reply:\n"
        '  {"thought": "<one short sentence>", "final_answer": "<your answer>", "evidence_path": "<the file you used>"}\n\n'
        "RULES: investigate step by step; use ONLY what the tools return; never invent paths, tables, "
        "or data; if a tool errors, try another path or tool; if the evidence is not there, say so "
        "plainly; always cite the exact file in evidence_path. Keep thoughts to one sentence. /no_think"
    )


def parse_action(raw):
    """Pull the JSON object out of the model's reply, tolerating <think> blocks, code fences, and
    prose before or after it (the first '{' that starts a valid JSON object wins)."""
    text = re.sub(r"<think>.*?</think>", "", raw or "", flags=re.S)
    text = text.replace("```json", "").replace("```", "")
    decoder = json.JSONDecoder()
    for m in re.finditer(r"\{", text):
        try:
            obj, _ = decoder.raw_decode(text, m.start())
        except ValueError:
            continue
        if isinstance(obj, dict):
            return obj
    return None


def chat(client, model, messages, max_tokens=None):
    """One model call. Some hosted models (gpt-oss on Groq) occasionally try a built-in function call
    instead of writing the JSON reply, which the provider rejects ("model called a tool"); when that
    happens, ask again with a plain-text reminder instead of stopping the investigation.
    max_tokens: 1,500 by default, 900 for Qwen (its free tier allows 1,000 reply tokens a minute)."""
    if max_tokens is None:
        max_tokens = 900 if "qwen" in str(model).lower() else 1500
    for attempt in range(3):
        try:
            return client.chat.completions.create(model=model, messages=messages, temperature=0,
                                                  max_tokens=max_tokens).choices[0].message.content or ""
        except Exception as e:
            if "called a tool" not in str(e) or attempt == 2:
                raise
            messages = messages + [{"role": "user", "content": "Do not use function calling. Reply with "
                                    "ONE JSON object written as plain text."}]


def run_agent_iter(question, client, model, tools, system_prompt=None, max_steps=12, history=None):
    """Generator version of the agent, for live UIs. Yields a dict per step
    ({step, thought, action, args, observation}) and finally the answer
    ({final_answer, evidence_path}) or a give-up dict ({final_answer: None}).

    history: optional prior [(user, assistant)] turns, so the agent remembers a conversation.
    """
    system_prompt = system_prompt or build_system_prompt(tools)
    messages = [{"role": "system", "content": system_prompt}]
    for u, a in (history or []):
        messages.append({"role": "user", "content": u})
        messages.append({"role": "assistant", "content": a})
    question = as_text(question)
    messages.append({"role": "user", "content": f"Investigate and answer this question: {question}"})

    seen = question + " " + " ".join(as_text(a) for _, a in (history or []))  # what it may rely on
    tools_run, pushbacks, ran = 0, 0, set()
    for step in range(1, max_steps + 1):
        raw = chat(client, model, messages)
        messages.append({"role": "assistant", "content": raw})
        move = parse_action(raw)
        if move is None:
            messages.append({"role": "user",
                             "content": "That was not valid JSON. Reply with ONE JSON object only."})
            continue
        if "final_answer" in move:
            answer = move["final_answer"]
            problem = evidence_check(answer, tools_run, seen, cited=move.get("evidence_path", ""),
                                     ran=ran, tool_names=list(tools))
            if problem and pushbacks < MAX_PUSHBACKS:
                # The harness, not the model, decides whether an answer is backed by evidence.
                pushbacks += 1
                yield {"step": step, "thought": "Evidence check: answer sent back",
                       "action": "evidence_check", "args": {}, "observation": problem}
                messages.append({"role": "user", "content": problem})
                continue
            if problem:   # still unsupported after the push-backs: deliver it, clearly flagged
                answer = f"{answer}\n\nUNVERIFIED: {problem}"
            yield {"final_answer": answer, "evidence_path": move.get("evidence_path", "")}
            return
        tool, args = move.get("action"), move.get("action_input", {}) or {}
        if tool not in tools:
            observation = f"Unknown tool {tool!r}. Use one of: {', '.join(tools)}."
            args = args if isinstance(args, dict) else {}
        else:
            if not isinstance(args, dict):   # e.g. "action_input": "data/data": give it to the 1st arg
                params = list(inspect.signature(tools[tool]).parameters)
                args = {params[0]: args} if params else {}
            try:
                observation = tools[tool](**args)
                tools_run += 1                 # only a tool that actually ran counts as evidence
                ran.add(tool)
            except Exception as e:
                observation = f"Tool error: {e}. Arguments are: {tool}{inspect.signature(tools[tool])}"
        seen += " " + str(observation)
        yield {"step": step, "thought": move.get("thought", ""),
               "action": tool, "args": args, "observation": observation}
        messages.append({"role": "user", "content": f"Observation:\n{observation}"})
    # Step limit reached. Do not throw away what was found: ask once for the best answer from the
    # observations so far, and run it through the same evidence check (flagged if it fails).
    if tools_run:
        messages.append({"role": "user", "content": "You have reached the step limit. Reply now with your "
                         "final_answer JSON, based only on the observations above. If they do not answer "
                         "the question, say so plainly."})
        move = parse_action(chat(client, model, messages))
        if move and "final_answer" in move:
            answer = move["final_answer"]
            problem = evidence_check(answer, tools_run, seen, cited=move.get("evidence_path", ""),
                                     ran=ran, tool_names=list(tools))
            if problem:
                answer = f"{as_text(answer)}\n\nUNVERIFIED: {problem}"
            yield {"final_answer": answer, "evidence_path": move.get("evidence_path", "")}
            return
    yield {"final_answer": None, "evidence_path": ""}


# ======================================================================= evidence check
# Limiting the tools to the evidence limits what the model can LOOK AT, not what it can SAY: a
# model can still write a plausible PID or file name from its training. So before an answer is
# accepted, the harness checks it: at least one tool must have run for this question, and every
# specific identifier in the answer must appear in something a tool actually returned (or in the
# question / earlier answers). Anything else is sent back to the model to confirm or remove.
MAX_PUSHBACKS = 2
EVIDENCE_PATTERNS = {
    "PID": r"\bPIDs?\s*[:#=]?\s*(\d{1,6})\b",
    "IP address": r"(?<![\d.])((?:\d{1,3}\.){3}\d{1,3})(?![\d.])",
    "memory address": r"\b(0x[0-9a-fA-F]{5,})\b",
    "file": r"\b([\w-]+\.(?:exe|dll|sys|apk|db|sqlite|xml|json|plist|log|txt|csv))\b",
}


def evidence_check(answer, tools_run, seen, cited="", ran=(), tool_names=()):
    """Return a push-back message if the answer is not backed by tool output, else None.
    cited / ran / tool_names: the answer's evidence line must not name a tool that never ran."""
    answer = as_text(answer)
    if tools_run == 0 and not answer.rstrip().endswith("?"):   # a question back to the user claims nothing
        return ("You answered without running any tool. Do not answer from memory or assumption: "
                "run the tool that shows this evidence, then answer only from its output.")
    seen_low = seen.lower()
    missing = []
    for kind, pattern in EVIDENCE_PATTERNS.items():
        for value in re.findall(pattern, str(answer), flags=re.I):
            found = (re.search(rf"(?<!\d){value}(?!\d)", seen) if kind == "PID"
                     else value.lower() in seen_low)
            if not found and f"{kind} {value}" not in missing:
                missing.append(f"{kind} {value}")
    if missing:
        return ("Your answer names " + ", ".join(missing[:8]) + ", which does not appear in any tool "
                "output you have seen. Run a tool that shows it, or remove it from your answer.")
    cited_text = as_text(cited)
    unrun = [t for t in tool_names if t not in ran and re.search(rf"\b{re.escape(t)}\b", cited_text)
             and not re.search(rf"\b{re.escape(t)}\b", seen)]
    if unrun:
        return ("Your evidence cites " + ", ".join(unrun) + ", which you have not run. Cite only the tools "
                "whose output you actually used, or run them first.")
    return None


def run_agent(question, client, model, tools, system_prompt=None, max_steps=12,
              on_step=None, history=None):
    """Drive the model think->act->observe until it answers.

    on_step(dict): optional callback for live UIs (gets each step dict, then the final dict).
    history: optional prior [(user, assistant)] turns, so the agent remembers a conversation.
    Returns the final move dict (with 'final_answer'/'evidence_path'), or None if it gave up.
    """
    final = None
    for ev in run_agent_iter(question, client, model, tools, system_prompt, max_steps, history):
        if on_step:
            on_step(ev)
        if "final_answer" in ev:
            final = ev if ev["final_answer"] is not None else None
    return final


def answer_text(move):
    """Format a final move as a human answer line with its citation."""
    if not move:
        return "I could not reach an answer within the step limit. Try rephrasing the question."
    cite = move.get("evidence_path") or "(no file cited)"
    return f"{move['final_answer']}\n\nSource: {cite}"


In [ ]:
%%writefile irkit.py
"""
irkit.py - Sectra First Responder: an incident-response copilot that works from a company's
own IR plan and playbooks.

DESIGN (the same discipline as the Sectra DF agent, applied to incident response)
---------------------------------------------------------------------------------
We took a comprehensive IR lifecycle (NIST SP 800-61 / SANS: identify -> notify -> collect ->
analyse -> contain -> eradicate -> recover -> after-action) and, for every action a playbook
typically calls for, decided what a language-model copilot can safely do, then built one tool
per action. Each tool is one of three kinds:

  RUN      - a real, local, read-only or analytical thing Sectra does itself (classify the
             incident, read/search/query artifacts, build a timeline, extract IOCs, check them
             against the watch-list, write the after-action report). No external side effects.
  DRAFT    - Sectra cannot touch production, so it produces ready-to-use content grounded in the
             playbook for a human to run: the notification email, the exact collection /
             containment / eradication / recovery commands and paths.
  SIMULATE - an action with an external side effect (notify management, page on-call, open a
             ticket, block an IOC, isolate a host). Sectra may only do it AFTER the team approves
             in chat, and even then it is simulated and written to the action log. Never real.

The agent reasons out loud (one short "thought" per step), uses exactly one tool per step, and
cites the plan/playbook section behind every instruction. Reuses the shared loop in agentkit.
"""
import os
import re
from pathlib import Path

import agentkit as ak

connect_model = ak.connect_model
run_agent = ak.run_agent
run_agent_iter = ak.run_agent_iter
answer_text = ak.answer_text

_HEADING = re.compile(r"^(#{1,3})\s+(.*)$")
_TEXT_EXT = {".txt", ".log", ".csv", ".json", ".xml", ".md", ".ini", ".conf", ".eml", ".tsv"}

# ---- indicator patterns (shared by extract_iocs / check_indicators) ----
_IPV4 = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")
_URL = re.compile(r"https?://[^\s\"'<>)]+", re.I)
_HASH = re.compile(r"\b[a-fA-F0-9]{64}\b|\b[a-fA-F0-9]{40}\b|\b[a-fA-F0-9]{32}\b")
_DOMAIN = re.compile(r"\b(?:[a-zA-Z0-9](?:[a-zA-Z0-9-]{0,61}[a-zA-Z0-9])?\.)+[a-zA-Z]{2,}\b")
_TS = re.compile(r"\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}")
# domain-looking tokens whose last label is really a file extension - not a domain
_NOT_TLD = {"exe", "dll", "txt", "log", "csv", "tsv", "7z", "zip", "pf", "hve", "raw", "evtx",
            "db", "sqlite", "json", "xml", "ini", "conf", "eml", "md", "py", "bat", "ps1", "sys"}


# ----------------------------------------------------------------- knowledge (plan + playbooks)
# Playbooks and IR plans arrive in several shapes. Real-world plans are usually PDF or Word;
# the bundled samples are Markdown. load_knowledge reads all of these, extracts plain text, and
# splits it into (section_title, body) pairs so the keyword tools work the same way on each.
_DOC_EXT = {".md", ".markdown", ".txt", ".pdf", ".docx"}

# section titles the tools key on - used to recognise headings in plain text (PDF/Word) that
# has no Markdown "##" marks. Keep this in sync with the tools' title keywords.
_KNOWN_HEADINGS = (
    "classification", "indicators", "immediate response", "first response", "response",
    "evidence collection", "evidence handling", "collection", "containment", "contain",
    "eradication", "recovery", "notification template", "management notification", "template",
    "roles", "severity", "scope", "purpose", "detection", "escalation", "communication",
    "legal", "regulatory", "after-action", "lessons", "decision", "mitre", "contact",
    "how to use", "overview", "summary", "preparation", "introduction",
)


def load_knowledge(folder):
    """Read every supported doc in `folder` (.md/.txt/.pdf/.docx) into
    {doc_name: [(section_title, body), ...]}. Files that yield no text are skipped."""
    knowledge = {}
    for p in sorted(Path(folder).iterdir()):
        if not (p.is_file() and p.suffix.lower() in _DOC_EXT):
            continue
        text = _read_doc_text(p)
        if text and text.strip():
            knowledge[p.name] = _split_sections(text)
    return knowledge


def _read_doc_text(path):
    """Return the plain text of a knowledge doc. PDF via pypdf, Word via python-docx, else read
    as text. Returns '' if it cannot be read (e.g. a scanned image-only PDF, or a missing lib)."""
    ext = Path(path).suffix.lower()
    if ext == ".pdf":
        return _pdf_text(path)
    if ext == ".docx":
        return _docx_text(path)
    try:
        return Path(path).read_text(errors="replace")
    except Exception:
        return ""


def _pdf_text(path):
    try:
        from pypdf import PdfReader
    except Exception:
        try:
            from PyPDF2 import PdfReader   # older name, same API
        except Exception:
            print(f"[{Path(path).name}: install pypdf to read PDFs - `pip install pypdf`. Skipped.]")
            return ""
    try:
        pages = [(pg.extract_text() or "") for pg in PdfReader(str(path)).pages]
        text = "\n".join(pages)
        if not text.strip():
            print(f"[{Path(path).name}: no extractable text (a scanned/image PDF needs OCR). Skipped.]")
        return text
    except Exception as e:
        print(f"[{Path(path).name}: could not read PDF ({e}). Skipped.]")
        return ""


def _docx_text(path):
    try:
        import docx   # python-docx
    except Exception:
        print(f"[{Path(path).name}: install python-docx to read Word files - `pip install python-docx`. Skipped.]")
        return ""
    try:
        return "\n".join(p.text for p in docx.Document(str(path)).paragraphs)
    except Exception as e:
        print(f"[{Path(path).name}: could not read DOCX ({e}). Skipped.]")
        return ""


def _looks_like_heading(line):
    """Heuristic heading test for plain text (PDF/Word) with no Markdown marks."""
    s = line.strip()
    if not s or len(s) > 80:
        return False
    low = s.lower().rstrip(":").strip()
    if any(low == h or low.startswith(h + " ") or low.startswith(h) and len(low) <= len(h) + 24
           for h in _KNOWN_HEADINGS):
        return True
    # numbered heading: "1. Title", "2.3 Containment"
    if re.match(r"^\d+(\.\d+)*[.)]?\s+\S", s) and len(s.split()) <= 10 and not s.endswith((".", "!", "?")):
        return True
    # a short ALL-CAPS line
    if s.isupper() and any(c.isalpha() for c in s) and len(s.split()) <= 10:
        return True
    return False


def _split_sections(text):
    """Split a document into (title, body) sections. Uses Markdown headings when present
    (the bundled .md files); falls back to a heading heuristic for plain text from PDF/Word."""
    lines = text.splitlines()
    has_md = any(_HEADING.match(ln) for ln in lines)

    def heading_of(line):
        if has_md:
            m = _HEADING.match(line)
            return m.group(2).strip() if m else None
        return line.strip().rstrip(":").strip() if _looks_like_heading(line) else None

    sections, title, cur = [], "(top)", []
    for line in lines:
        h = heading_of(line)
        if h is not None:
            if cur:
                sections.append((title, "\n".join(cur).strip()))
            title, cur = h, []
        else:
            cur.append(line)
    sections.append((title, "\n".join(cur).strip()))
    return [(t, b) for t, b in sections if b or t != "(top)"]


def _domains(text):
    out = []
    for d in _DOMAIN.findall(text or ""):
        if d.split(".")[-1].lower() in _NOT_TLD:
            continue
        if _IPV4.fullmatch(d):
            continue
        out.append(d.lower())
    return out


# ----------------------------------------------------------------- the tools
def make_ir_tools(knowledge, evidence_root, action_log, org="SectraTEK"):
    """Build the full IR toolbox. `action_log` is a list the SIMULATE/RUN tools append to."""
    EV = Path(evidence_root)
    EV.mkdir(parents=True, exist_ok=True)

    # ============================================================ shared helpers
    def _all_sections():
        for doc, secs in knowledge.items():
            for title, body in secs:
                yield doc, title, body

    def _score(words, doc, text):
        """How well a playbook section fits the goal: word hits, with hits in the file name x3."""
        return sum(1 for w in words if w in text) + 3 * sum(1 for w in words if w in doc.lower())

    def _pull(title_keywords, goal="", limit=2000):
        """Collect playbook sections whose title matches any keyword, best fit to the goal first
        (by the goal's words, so 'ransomware on the finance PC' finds the ransomware playbook)."""
        words = set(re.findall(r"[a-z]{4,}", (goal or "").lower()))
        scored = []
        for doc, title, body in _all_sections():
            if any(w in title.lower() for w in title_keywords):
                score = _score(words, doc, (title + " " + body).lower())
                scored.append((score, f"[{doc} > {title}]\n{body}"))
        if not scored:
            return None
        scored.sort(key=lambda x: -x[0])          # stable: equal scores keep document order
        best = scored[0][0]
        chosen = [b for sc, b in scored if sc == best] if best > 0 else [b for _, b in scored]
        text = "\n\n".join(chosen)
        return text[:limit] + ("" if len(text) <= limit else "\n...[truncated]")

    def _known_bad():
        """Parse the threat-intel watch-list from any knowledge section about indicators."""
        items = []
        for doc, title, body in _all_sections():
            if not any(w in title.lower() for w in ("indicator", "known-bad", "known bad",
                                                    "watch-list", "watchlist", "threat intel", "ioc")):
                continue
            for line in body.splitlines():
                line = line.strip().lstrip("-*").strip()
                if not line:
                    continue
                parts = re.split(r"\s{2,}", line)
                val = parts[0].strip()
                note = "  ".join(p.strip() for p in parts[1:]) if len(parts) > 1 else ""
                if val and " " not in val:
                    items.append((val, note, doc, title))
        return items

    def _safe(path):
        base = EV.resolve()
        path = (path or ".").strip().lstrip("/")
        t = base if path in ("", ".") else (EV / path).resolve()
        if t != base and base not in t.parents:
            raise ValueError("path is outside the evidence folder")
        return t

    def _read_text(path):
        t = _safe(path)
        if not t.exists() or t.is_dir():
            return None, f"No such artifact: {path}"
        if ak.read_head(t)[:15] == b"SQLite format 3":
            return None, f"[{path} is a SQLite database. Use run_sql with a SELECT.]"
        return t.read_text(errors="replace"), None

    # ============================================================ 1. KNOWLEDGE (RUN, read-only)
    def list_playbooks():
        """RUN: list the IR plan and playbooks with their section titles. Start here to see what exists."""
        out = []
        for doc, secs in knowledge.items():
            out.append(doc + ":")
            out += [f"    - {t}" for t, _ in secs]
        return "\n".join(out) if out else "(no playbooks loaded)"

    def search_playbooks(keyword):
        """RUN: search the plan and playbooks for a keyword; returns [doc > section] matching lines."""
        key = (keyword or "").lower()
        out = []
        for doc, title, body in _all_sections():
            for line in body.splitlines():
                if key in line.lower():
                    out.append(f"[{doc} > {title}] {line.strip()[:140]}")
                    if len(out) >= 25:
                        return "\n".join(out) + "\n...[more]"
        return "\n".join(out) if out else f"No playbook text matches {keyword!r}."

    def read_playbook(name, section=""):
        """RUN: read a playbook (by file name) or one of its sections (by text in the title)."""
        want_doc, want_sec = (name or "").lower(), (section or "").lower()
        hits = []
        for doc, title, body in _all_sections():
            if want_doc and want_doc not in doc.lower():
                continue
            if want_sec and want_sec not in title.lower():
                continue
            hits.append(f"[{doc} > {title}]\n{body}")
        if not hits:
            return f"No playbook/section matching name={name!r} section={section!r}. Use list_playbooks."
        text = "\n\n".join(hits)
        return text[:2200] + ("" if len(text) <= 2200 else "\n...[truncated]")

    def classify_incident(description):
        """RUN: match the team's description to the best-fitting playbook and its severity, and
        return the playbook's first response step. Use this at the very start of an incident."""
        words = set(re.findall(r"[a-z]{4,}", (description or "").lower()))
        best, best_score = None, 0
        for doc, secs in knowledge.items():
            if "playbook" not in doc.lower():   # only incident playbooks are classified (not plan/ISMS/intel)
                continue
            text = " ".join(t + " " + b for t, b in secs).lower()
            score = sum(1 for w in words if w in text)
            # a direct hit on the incident name in the filename is worth a lot
            score += 5 * sum(1 for w in words if w in doc.lower())
            if score > best_score:
                best, best_score = doc, score
        if not best:
            return "Could not match a playbook. Use list_playbooks and ask the team to clarify the incident."
        sev = next((b for t, b in knowledge[best] if "classif" in t.lower()), "(severity not stated)")
        first = next((b for t, b in knowledge[best] if any(w in t.lower()
                     for w in ("immediate", "first", "response"))), "(no immediate-response section)")
        return (f"Best-matching playbook: {best}\n\nClassification / severity:\n{sev}\n\n"
                f"First response steps:\n{first}\n\n"
                "Next: if a specific host, account, or system was named, call lookup_asset to get its "
                "criticality and classification from the ISMS, and let that confirm the severity.")

    # ============================================================ 2. NOTIFY / COORDINATE
    def draft_notification(audience="management", incident=""):
        """DRAFT: write a notification for `audience` (management/legal/customers/regulator), pulling
        the template from the playbook if there is one. This DRAFTS only - show it and ask to approve."""
        words = set(re.findall(r"[a-z]{4,}", (incident or "").lower()))
        templates = [(_score(words, doc, title.lower()), doc, title, body)
                     for doc, title, body in _all_sections() if "template" in title.lower()]
        templates = [t for t in templates if t[0] > 0]
        if templates:
            _, doc, title, body = max(templates, key=lambda t: t[0])
            return f"(template from [{doc} > {title}], audience={audience})\n{body}"
        return (f"To: {audience}\nSubject: Security incident in progress ({incident or 'under triage'})\n\n"
                f"Team, we are responding to a suspected {incident or 'security incident'} at {org}. The "
                "response team is engaged and following the IR plan. We will send an update within 60 "
                "minutes or on any material change. [No playbook template found; generic draft.]")

    def _approved(value):
        """Strict approval check: only a real yes counts ("false", "no" or "" do not)."""
        return value is True or str(value).strip().lower() in ("true", "yes", "approved", "1")

    def notify(audience="management", approved=False):
        """SIMULATE (gated): 'send' the notification to `audience`. Only after the team approves in
        chat (approved=True). Nothing is really sent; it is logged to the action log."""
        if not _approved(approved):
            return "Not sent. I need the team's explicit approval first (reply 'approve' to send)."
        entry = f"[DRILL/SIMULATED] Notification 'sent' to {audience}."
        action_log.append(entry)
        return entry + " (No real email was sent - training drill.)"

    def open_ticket(summary="", approved=False):
        """SIMULATE (gated): open an incident ticket. Only with the team's approval. Logged, not real."""
        if not _approved(approved):
            return "Not opened. Confirm with the team first (approved=true) and I will log it."
        entry = f"[DRILL/SIMULATED] Incident ticket opened: {summary or 'security incident'}."
        action_log.append(entry)
        return entry

    def page_oncall(role="IT on-call", approved=False):
        """SIMULATE (gated): page an on-call role (e.g. 'IT on-call', 'Forensics Lead'). Gated + logged."""
        if not _approved(approved):
            return f"Not paged. Confirm with the team first (approved=true) to page {role}."
        entry = f"[DRILL/SIMULATED] Paged {role}."
        action_log.append(entry)
        return entry

    # ============================================================ 3. GUIDANCE (DRAFT, human runs it)
    def recommend_collection(goal=""):
        """DRAFT: the exact tools, commands, and paths to COLLECT evidence, from the playbook."""
        out = _pull(("collect", "evidence", "acquire", "preserve"), goal)
        if out:
            action_log.append(f"Recommended evidence-collection steps for: {goal or 'the incident'}.")
        return out or "No collection section matched. Try read_playbook(section='collection')."

    def recommend_containment(goal=""):
        """DRAFT: the CONTAINMENT steps/commands from the playbook (isolate host, disable account,
        block C2). The team runs these; Sectra can only simulate them via contain()."""
        out = _pull(("contain", "isolat"), goal)
        return out or "No containment section matched. Try read_playbook(section='contain')."

    def recommend_eradication(goal=""):
        """DRAFT: the ERADICATION steps from the playbook (remove persistence/malware, reset creds)."""
        out = _pull(("eradicat", "remov", "rebuild", "contain"), goal)
        return out or "No eradication section matched. Try read_playbook(section='eradicat')."

    def recommend_recovery(goal=""):
        """DRAFT: the RECOVERY steps from the playbook (restore from clean backup, validate, monitor)."""
        out = _pull(("recover", "restore", "monitor"), goal)
        return out or "No recovery section matched. Try read_playbook(section='recover')."

    # ============================================================ 4. CONTAINMENT ACTIONS (SIMULATE)
    def contain(kind="isolate_host", target="", approved=False):
        """SIMULATE (gated): take a containment action - kind is 'isolate_host', 'disable_account',
        or 'block_ioc'; target is the host/account/indicator. Only with approval; logged, not real."""
        done = {"isolate_host": "Isolated host", "disable_account": "Disabled account",
                "block_ioc": "Blocked indicator at the firewall"}
        plan = {"isolate_host": "isolate the host", "disable_account": "disable the account",
                "block_ioc": "block the indicator at the firewall"}
        if not _approved(approved):
            intent = plan.get(kind, f"take containment action '{kind}' on")
            return f"Not done. This would {intent}: {target or '(target?)'}. Get the team's approval first."
        entry = f"[DRILL/SIMULATED] {done.get(kind, kind)}: {target or '(unspecified)'}."
        action_log.append(entry)
        return entry

    # ============================================================ 5. EVIDENCE ANALYSIS (RUN)
    def list_evidence(path="."):
        """RUN: list the artifacts the team has collected into the evidence folder."""
        t = _safe(path)
        if not t.exists():
            return "(no evidence collected yet)"
        if t.is_file():
            return f"{path} is a file. Use read_evidence, run_sql, build_timeline, or extract_iocs."
        items = [e + ("/" if (t / e).is_dir() else "") for e in sorted(os.listdir(t))]
        return "\n".join(items) if items else "(no evidence collected yet)"

    def read_evidence(path, max_chars=1600):
        """RUN: read a collected text/log artifact (or get a hint for a SQLite database)."""
        text, err = _read_text(path)
        if err:
            return err
        return text[:max_chars] + ("" if len(text) <= max_chars else "\n...[truncated]")

    def search_evidence(keyword, max_hits=25):
        """RUN: search the collected artifacts for a keyword; returns artifact:line matches."""
        base = EV.resolve()
        key = (keyword or "").lower()
        out = []
        for p in sorted(base.rglob("*")):
            if p.is_symlink() or not p.is_file() or p.suffix.lower() not in _TEXT_EXT:
                continue
            for i, line in enumerate(p.read_text(errors="replace").splitlines(), 1):
                if key in line.lower():
                    out.append(f"{p.relative_to(base)}:{i}: {line.strip()[:140]}")
                    if len(out) >= max_hits:
                        return "\n".join(out) + "\n...[more]"
        return "\n".join(out) if out else f"No evidence matches {keyword!r}."

    def run_sql(path, query):
        """RUN: run ONE read-only SELECT against a collected SQLite artifact."""
        import sqlite3
        t = _safe(path)
        if not t.exists():
            return f"No such artifact: {path}"
        q = (query or "").strip().rstrip(";")
        if not q.lower().startswith(("select", "pragma")):
            return "Only read-only SELECT queries are allowed."
        try:
            with ak.sqlite_copy(t) as con:   # query a temporary copy; the artifact is never opened
                cur = con.execute(q)
                cols = [c[0] for c in cur.description] if cur.description else []
                rows = cur.fetchmany(50)
        except Exception as e:
            return f"SQL error: {e}"
        if not rows:
            return "(no rows)"
        return "\n".join([" | ".join(cols)] + [" | ".join(str(c) for c in r) for r in rows])

    def build_timeline(path, max_rows=40):
        """RUN: build a time-ordered timeline from a log/CSV artifact (lines carrying a timestamp)."""
        text, err = _read_text(path)
        if err:
            return err
        rows = []
        for line in text.splitlines():
            m = _TS.search(line)
            if m:
                rows.append((m.group(0).replace("T", " "), line.strip()))
        if not rows:
            return f"No timestamped lines found in {path}."
        rows.sort(key=lambda r: r[0])
        shown = rows[:max_rows]
        body = "\n".join(f"{ts}  {ln[:150]}" for ts, ln in shown)
        more = "" if len(rows) <= max_rows else f"\n...[{len(rows) - max_rows} more events]"
        return f"Timeline of {path} ({len(rows)} events):\n{body}{more}"

    def extract_iocs(path):
        """RUN: extract indicators (IPs, domains, URLs, file hashes) from a collected artifact."""
        text, err = _read_text(path)
        if err:
            return err
        ips = sorted(set(_IPV4.findall(text)))
        doms = sorted(set(_domains(text)))
        urls = sorted(set(_URL.findall(text)))
        hashes = sorted(set(_HASH.findall(text)))
        if not any([ips, doms, urls, hashes]):
            return f"No IPs, domains, URLs, or hashes found in {path}."
        parts = []
        if ips: parts.append("IPs:\n  " + "\n  ".join(ips[:30]))
        if doms: parts.append("Domains:\n  " + "\n  ".join(doms[:30]))
        if urls: parts.append("URLs:\n  " + "\n  ".join(urls[:20]))
        if hashes: parts.append("Hashes:\n  " + "\n  ".join(hashes[:20]))
        return f"Indicators in {path}:\n" + "\n".join(parts) + \
               "\n(Use check_indicators to compare these against the watch-list.)"

    def check_indicators(path="", values=""):
        """RUN: compare indicators against the threat-intel watch-list. Give `path` (an artifact to
        scan) and/or `values` (comma/space-separated indicators). Returns any known-bad matches."""
        cand = set()
        if path:
            text, err = _read_text(path)
            if err:
                return err
            cand |= set(_IPV4.findall(text)) | set(_domains(text)) | set(x.lower() for x in _HASH.findall(text))
            for line in text.splitlines():  # bare file/service names too
                cand |= set(re.findall(r"\b[\w.-]+\.(?:exe|dll|txt|sys)\b", line, re.I))
                cand |= set(re.findall(r"name=([\w.-]+)", line))
        if values:
            cand |= {v.strip() for v in re.split(r"[\s,]+", values) if v.strip()}
        cand = {c.lower() for c in cand if c}
        if not cand:
            return "No indicators to check. Give a `path` to an artifact or a `values` list."
        matches = []
        for val, note, doc, title in _known_bad():
            v = val.lower()
            for c in cand:
                # exact match, or a subdomain of a known-bad domain (cdn.bad.example matches bad.example)
                if c == v or (not _IPV4.fullmatch(v) and c.endswith("." + v)):
                    matches.append(f"MATCH {val}  (seen in {c})  - {note} [{doc} > {title}]")
                    break
        if not matches:
            return (f"Checked {len(cand)} indicator(s) against the watch-list: no known-bad matches. "
                    "That does not clear them - it means none are on our small list.")
        action_log.append(f"Watch-list hits: {len(matches)} indicator(s) matched known-bad.")
        return "Known-bad matches:\n" + "\n".join(matches)

    # ============================================================ 6. ISMS / ISO 27001 CONTEXT (RUN)
    # The copilot can be given the company's ISO 27001 ISMS documents (asset register, risk register,
    # classification and incident-management policies, Statement of Applicability). These tools read
    # that context so the response stays aligned with the business value of the asset and company policy.
    def _isms_docs(kinds):
        """Yield (doc, sections) for ISMS docs whose file name matches any keyword in `kinds`."""
        for doc, secs in knowledge.items():
            if any(k in doc.lower() for k in kinds):
                yield doc, secs

    def _md_rows(doc_kw):
        """Parse the markdown table in the ISMS doc whose name matches doc_kw into dict rows."""
        rows = []
        for doc, secs in _isms_docs((doc_kw,)):
            header = None
            for _, body in secs:
                for line in body.splitlines():
                    line = line.strip()
                    if not line.startswith("|"):
                        continue
                    cells = [c.strip() for c in line.strip("|").split("|")]
                    if set("".join(cells)) <= set("-: "):   # the |---| separator row
                        continue
                    if header is None:
                        header = cells
                        continue
                    if len(cells) == len(header):
                        rows.append((doc, dict(zip(header, cells))))
        return rows

    def _asset_table():
        return _md_rows("asset")

    def lookup_asset(name=""):
        """RUN: look up a host/account/system in the ISMS asset register by name or alias, and return
        its owner, classification, criticality, business function and RTO/RPO. Call this whenever the
        team names a specific asset, so severity and recovery priority match its business value."""
        q = (name or "").strip().lower()
        rows = _asset_table()
        if not rows:
            return "No asset register is loaded. Add the ISMS asset register to the knowledge folder."
        if not q:
            return list_assets()
        def _alias_of(r):
            return (next((v for k, v in r.items() if "alias" in k.lower()), "")
                    or next((v for k, v in r.items() if "asset" in k.lower() and "id" not in k.lower()), ""))
        hits = []
        for doc, r in rows:
            idc = next((v for k, v in r.items() if "id" in k.lower()), "")
            alias = _alias_of(r)
            hay = (idc + " | " + alias).lower()
            parts = [p.strip().lower() for p in re.split(r"[;,]", alias)] + [idc.lower()]
            if q in hay or any(q in p or (len(p) > 3 and p in q) for p in parts if p):
                hits.append((doc, r))
        if not hits:
            return (f"'{name}' is not in the asset register. Ask the owning team for its classification "
                    "and criticality; do not guess. Use lookup_asset() with no name to list known assets.")
        out = []
        for doc, r in hits[:3]:
            fields = " | ".join(f"{k}: {v}" for k, v in r.items())
            out.append(f"[{doc}]\n{fields}")
        return "\n\n".join(out)

    def list_assets():
        """RUN: list the assets in the ISMS asset register with their criticality and classification."""
        rows = _asset_table()
        if not rows:
            return "No asset register is loaded."
        out = []
        for doc, r in rows:
            idc = next((v for k, v in r.items() if "id" in k.lower()), "?")
            alias = (next((v for k, v in r.items() if "alias" in k.lower()), "")
                     or next((v for k, v in r.items() if "asset" in k.lower() and "id" not in k.lower()), ""))
            crit = next((v for k, v in r.items() if "critical" in k.lower()), "?")
            cls = next((v for k, v in r.items() if "classif" in k.lower()), "?")
            out.append(f"- {idc}: {alias.split(';')[0].strip()}  (criticality: {crit}, {cls})")
        return "Assets in the register:\n" + "\n".join(out)

    def check_policy(topic=""):
        """RUN: find the relevant ISMS policy/control for a topic (e.g. 'breach notification',
        'evidence', 'classification', 'backup', 'risk'), so the response stays within company policy."""
        words = [w for w in re.findall(r"[a-z]{3,}", (topic or "").lower())]
        # policy / control / risk docs only; asset data is served by lookup_asset / list_assets
        kinds = ("policy", "classification", "incident_management", "statement",
                 "applicability", "soa", "risk_register")
        scored = []
        for doc, secs in _isms_docs(kinds):
            for title, body in secs:
                tl, bl = title.lower(), body.lower()
                if not words:
                    scored.append((1, doc, title, body)); continue
                s = sum(3 for w in words if w in tl) + sum(1 for w in words if w in bl)
                if s:
                    scored.append((s, doc, title, body))
        if not scored:
            docs = [d for d, _ in _isms_docs(kinds)]
            return (f"No ISMS section matched {topic!r}. ISMS documents loaded: "
                    f"{', '.join(docs) if docs else '(none)'}.")
        scored.sort(key=lambda h: -h[0])
        out = [f"[{d} > {t}]\n{b}" for _, d, t, b in scored[:2]]
        text = "\n\n".join(out)
        return text[:2000] + ("" if len(text) <= 2000 else "\n...[truncated]")

    def lookup_risk(query=""):
        """RUN: find the ISMS risk-register entry for an asset or incident type (e.g. 'finance
        workstation', 'FIN-WS-04', or 'ransomware') and return its rating and treatment, so the team
        can see whether a planned control held or failed and feed that into the after-action review."""
        rows = _md_rows("risk_register")
        if not rows:
            return "No risk register is loaded. Add the ISMS risk register to the knowledge folder."
        q = (query or "").strip().lower()
        # bridge an asset name/alias to its asset IDs, so 'finance workstation' matches rows naming FIN-WS-04
        tokens = {q} if q else set()
        for _, a in _md_rows("asset"):
            idc = next((v for k, v in a.items() if "id" in k.lower()), "")
            alias = (next((v for k, v in a.items() if "alias" in k.lower()), "")
                     or next((v for k, v in a.items() if "asset" in k.lower() and "id" not in k.lower()), ""))
            parts = [p.strip().lower() for p in re.split(r"[;,]", alias)]
            if q and (q in alias.lower() or any(q in p or (len(p) > 3 and p in q) for p in parts if p)):
                tokens.add(idc.lower())
        hits = []
        for doc, r in rows:
            hay = " ".join(r.values()).lower()
            if not q or any(t and t in hay for t in tokens):
                hits.append((doc, r))
        if not hits:
            return (f"No risk-register entry matched {query!r}. Use check_policy('risk') to see the "
                    "register, or ask the risk owner.")
        out = []
        for doc, r in hits[:4]:
            fields = " | ".join(f"{k}: {v}" for k, v in r.items())
            out.append(f"[{doc}]\n{fields}")
        return "\n\n".join(out)

    # ============================================================ 7. POST-INCIDENT (RUN)
    def after_action_report():
        """RUN: produce a short after-action report from the action log and the incident so far."""
        log = action_log or []
        lines = [f"After-action report (DRILL) - {org}", "=" * 40,
                 f"Actions taken / recommended ({len(log)}):"]
        lines += [f"  {i}. {e}" for i, e in enumerate(log, 1)] or ["  (none logged yet)"]
        lines += ["", "Reminder: write the full timeline and improvements within 5 business days "
                      "(see the IR plan, Lessons learned). All notifications above were simulated."]
        action_log.append("Generated after-action report.")
        return "\n".join(lines)

    return {
        # knowledge
        "list_playbooks": list_playbooks, "search_playbooks": search_playbooks,
        "read_playbook": read_playbook, "classify_incident": classify_incident,
        # notify / coordinate
        "draft_notification": draft_notification, "notify": notify,
        "open_ticket": open_ticket, "page_oncall": page_oncall,
        # guidance (draft)
        "recommend_collection": recommend_collection, "recommend_containment": recommend_containment,
        "recommend_eradication": recommend_eradication, "recommend_recovery": recommend_recovery,
        # containment actions (simulate)
        "contain": contain,
        # evidence analysis (run)
        "list_evidence": list_evidence, "read_evidence": read_evidence,
        "search_evidence": search_evidence, "run_sql": run_sql,
        "build_timeline": build_timeline, "extract_iocs": extract_iocs,
        "check_indicators": check_indicators,
        # ISMS / ISO 27001 context (run)
        "lookup_asset": lookup_asset, "list_assets": list_assets,
        "lookup_risk": lookup_risk, "check_policy": check_policy,
        # post-incident
        "after_action_report": after_action_report,
    }


def build_ir_system_prompt(tools, org="SectraTEK"):
    return (
        f"You are Sectra, the incident first responder for {org}. A response team has called you about a "
        "possible security incident. You work ONLY from the company's incident-response plan, playbooks, "
        "and ISO 27001 ISMS documents (the asset register, risk register, and policies) - your knowledge "
        "tools - plus the artifacts the team collects. This is a training DRILL: anything with an external "
        "effect is simulated, never real.\n\n"
        "WHAT YOU CAN DO - every tool is one of three kinds:\n"
        "  RUN      - you do it yourself (classify, read/search/query artifacts, build a timeline, extract "
        "and check indicators, write the after-action report).\n"
        "  DRAFT    - you cannot touch production, so you produce ready-to-use content for a human to run "
        "(the notification text, and the exact collection / containment / eradication / recovery commands).\n"
        "  SIMULATE - an action with a side effect (notify, open_ticket, page_oncall, contain). You may ONLY "
        "do it after the team approves in chat (approved=true); it is then logged, not real.\n\n"
        "TOOLS (use exactly one per step):\n" + ak.tool_descriptions(tools) + "\n\n"
        "HOW TO REPLY: send ONE JSON object and nothing else.\n"
        "To use a tool:\n"
        '  {"thought": "<one short sentence of reasoning>", "action": "classify_incident", "action_input": {"description": "..."}}\n'
        "To speak to the team (give a step, ask a question, or ask for approval):\n"
        '  {"thought": "<one short sentence>", "final_answer": "<what you tell the team>", "evidence_path": "<the plan/playbook section you used>"}\n\n'
        "HOW TO HANDLE A TURN:\n"
        "1. Work out what the plan/playbooks SAY about the team's situation (classify_incident, "
        "search_playbooks, read_playbook) and tell them, in plain words, citing the section.\n"
        "2. If a specific host, account, or system is named, call lookup_asset to get its criticality and "
        "classification from the ISMS, and let that confirm or raise the severity and set the recovery "
        "priority. Use check_policy to keep your guidance inside company policy (classification, breach "
        "notification, evidence). Cite the ISMS document just like a playbook. Never recommend anything "
        "that conflicts with the ISMS, and do not guess an asset's criticality if it is not in the register.\n"
        "3. Then OFFER the concrete next action you can take for them, and say which kind it is. RUN analysis "
        "yourself when artifacts are present. DRAFT content the team must run. For any SIMULATE action, show "
        "what you will do and ASK for approval first; only call it with approved=true after they agree.\n"
        "4. Follow the incident through its phases as the playbook dictates: identify -> notify management -> "
        "collect -> analyse the collected artifacts -> contain -> eradicate -> recover -> after-action.\n\n"
        "RULES: base every instruction on the plan/playbooks and name the section in evidence_path; do only "
        "one thing per step; never invent commands, data, or indicators; keep replies short and actionable; "
        "never take a SIMULATE action without explicit approval in the chat. /no_think"
    )


def quick_agent(message, client, model, tools, history=None, **kw):
    prompt = build_ir_system_prompt(tools)
    return run_agent(message, client, model, tools, prompt, history=history, **kw)


In [ ]:
%%writefile knowledge/00_IR_Plan.md
# SectraTEK - Incident Response Plan (SYNTHETIC, for training)

Document owner: SectraTEK Security Operations Centre (SOC). Version 3.0. Review cycle: every 6 months
and after every SEV-1. This plan is fictional and exists only for the Sectra First Responder lab.

## Purpose and scope
This plan governs how SectraTEK detects, responds to, and recovers from security incidents affecting
corporate endpoints, servers, mobile devices, and cloud/identity services. It applies to all staff,
contractors, and automated systems. It defines severity, roles, the response lifecycle, escalation and
notification rules, evidence handling, and legal obligations. The goal is a response that is fast,
consistent, and defensible: every decision is recorded and every piece of evidence is preserved so the
scope can be measured and, if needed, presented to auditors, regulators, or a court.

## Severity levels and response times
Severity is assigned by the first responder using these definitions and is re-checked as facts change.
- SEV-1 (Critical): active ransomware, confirmed data theft, business-down outage, or any confirmed
  compromise of a privileged/admin account. Engage the Incident Commander and notify management
  immediately. Target: acknowledge in 15 minutes, response team engaged in 30 minutes.
- SEV-2 (High): malware on a single host, confirmed phishing with clicks or entered credentials,
  single-user account compromise, suspected data exposure of limited scope. Target: acknowledge in 1
  hour, contain within the business day.
- SEV-3 (Medium): a reported suspicious email with no clicks, one blocked alert, a policy violation, a
  single failed-login spike that was auto-blocked. Target: triage within 1 business day.
- SEV-4 (Low / informational): near-miss, scan noise, or a finding raised for awareness. Log and review.
When unsure between two levels, choose the higher one and downgrade later with a documented reason.

## Roles and responsibilities (RACI)
- Incident Commander (IC): owns the response and all decisions; declares severity; the only role that
  can authorise containment of production systems. Accountable for the whole incident.
- SOC Analyst / First Responder: triages the alert, classifies the incident, collects evidence, runs
  the analysis. Responsible for the hands-on work.
- Forensics Lead: preserves and analyses host, memory, and mobile evidence; maintains chain of custody.
- Communications Lead: drafts and sends management, customer, and (with Legal) regulator messaging.
- IT On-call: performs containment actions the IC approves (isolate host, disable account, block IOC).
- Legal and Privacy Counsel: decides breach-notification and regulatory obligations; consulted on SEV-1.
- Executive Sponsor (CISO): informed on every SEV-1; approves external/customer notification.
Consulted: Legal, HR (insider cases), Comms. Informed: affected business owners, Executive Sponsor.

## The response lifecycle (NIST SP 800-61 / SANS PICERL)
1. Preparation: maintain this plan, the playbooks, logging, backups, and on-call rotas; run drills.
2. Detection and analysis: confirm the alert is a real incident, classify it, and scope it from evidence.
3. Containment: limit the damage. Short-term (isolate the host now) then long-term (rebuild clean).
4. Eradication: remove the root cause - malware, persistence, and attacker access.
5. Recovery: restore from known-clean backups, validate, and monitor for recurrence before closing.
6. Post-incident: write the after-action report; capture lessons learned; update controls and playbooks.
Evidence is preserved before any destructive containment, so scope can always be reconstructed.

## Escalation and notification matrix
- SEV-1: IC + CISO immediately; management within 30 minutes; Legal engaged; Comms on standby for
  customer/regulator messaging. Use the notification template in the relevant playbook.
- SEV-2: IC and SOC lead; management notified within 2 hours if customer data or a privileged account
  is involved; otherwise summarised in the daily report.
- SEV-3 / SEV-4: handled by the SOC; summarised in the daily and weekly reports. No management page.
Escalate upward whenever scope, data sensitivity, or business impact grows. Never wait for certainty to
escalate a SEV-1.

## Communications protocol
Use the out-of-band channel (the incident bridge and the SOC phone tree), never the potentially
compromised corporate email or chat, until containment is confirmed. One Communications Lead speaks for
the incident. Hold external statements until Legal and the CISO approve. Record every notification (who,
what, when) in the incident ticket.

## Evidence handling and chain of custody
Preserve before you change: capture volatile data (memory, network state) and logs before isolating or
rebuilding. Record for each artifact: what it is, where and when it was collected, by whom, and its hash.
Store evidence read-only in the case folder; work from copies. Do not power off a host under active
ransomware (memory is evidence) unless the IC directs it. Maintain an unbroken custody log so the
evidence is admissible.

## Legal and regulatory obligations
Confirmed theft or exposure of personal data triggers breach-notification duties with hard deadlines
(for example, many regimes require regulator notification within 72 hours of becoming aware). Legal
owns this decision; the SOC's job is to give Legal an accurate, evidenced scope quickly. Preserve
evidence to the legal standard from the first minute, because an incident can become litigation.

## Authority to act
The IC authorises containment of production systems. The SOC may take read-only investigative steps at
any time without further approval. Any action with an external side effect (notifying people, opening
tickets, paging on-call, blocking at the firewall, disabling accounts) requires explicit approval and
is logged. Automated copilots may draft and recommend freely but must not act without a human's yes.

## Drill cadence and maintenance
Run a tabletop drill quarterly and a technical drill twice a year. After every SEV-1 and every drill,
update this plan and the playbooks within two weeks. Keep the contact tree and tool list current.

## Contact tree (synthetic)
- SOC / first responders: soc@sectratek.example
- Incident Commander on-call: ic@sectratek.example
- Forensics Lead: forensics@sectratek.example
- Communications Lead: comms@sectratek.example
- Legal and Privacy: legal@sectratek.example
- Management distribution: management@sectratek.example


In [ ]:
%%writefile knowledge/isms_asset_register.md
# SectraTEK - ISMS Asset Register and Business Impact (SYNTHETIC, for training)

Part of the SectraTEK ISO/IEC 27001 Information Security Management System (ISMS). This register records
information assets, their owners, classification, criticality, and recovery objectives. The incident
copilot consults it so its guidance matches the business value of the asset involved. All data is
fictional and exists only for the Sectra First Responder lab.

## Classification and criticality scale
- Classification (confidentiality): Restricted > Confidential > Internal > Public.
- Criticality (business impact if lost or down): Critical > High > Medium > Low.
- RTO = recovery time objective (how fast it must be back). RPO = recovery point objective (tolerable
  data loss). MTD = maximum tolerable downtime.

## Asset register
Each row: Asset ID | Asset and aliases | Owner | Classification | Criticality | Business function | Location | RTO / RPO

| Asset ID | Asset and aliases | Owner | Classification | Criticality | Business function | Location | RTO / RPO |
|---|---|---|---|---|---|---|---|
| FIN-WS-04 | Finance workstation; finance PC; AP workstation; accounts payable PC | Finance Operations | Confidential | High | Accounts-payable and payroll preparation | HQ, Finance floor | 4h / 1h |
| DC-01 | Domain controller; Active Directory; AD | IT Infrastructure | Restricted | Critical | Identity and authentication for all staff | HQ data centre | 1h / 15m |
| DB-CUST-01 | Customer database; CRM database; customer data server | Data Platform | Restricted | Critical | Holds customer PII and account records | Cloud (primary region) | 2h / 15m |
| PAY-GW-01 | Payment gateway; billing gateway | Finance Systems | Restricted | Critical | Processes customer card payments | Cloud (PCI segment) | 1h / 5m |
| MAIL-365 | Email; Microsoft 365 tenant; Exchange Online | IT Infrastructure | Confidential | High | Corporate email and collaboration | SaaS (Microsoft 365) | 4h / 1h |
| FS-01 | File server; shared drive; departmental shares | IT Infrastructure | Confidential | High | Shared documents incl. Finance and HR folders | HQ data centre | 8h / 4h |
| BKP-01 | Backup server; backup repository | IT Infrastructure | Confidential | High | Offline and immutable backups for all systems | HQ data centre (segmented) | 8h / 24h |
| VPN-GW-01 | VPN gateway; remote-access gateway | IT Infrastructure | Confidential | High | Remote staff access | HQ edge | 4h / n/a |
| HR-APP-01 | HR system; HRIS | Human Resources | Restricted | High | Employee records and payroll data | SaaS | 24h / 8h |
| EXEC-LT-02 | CEO laptop; executive laptop | IT Infrastructure | Restricted | High | Executive device, high-value phishing target | Mobile | 8h / 4h |
| WEB-01 | Public website; marketing site | Marketing | Public | Medium | Public web presence | Cloud (DMZ) | 12h / 12h |
| KIOSK-03 | Reception kiosk; lobby kiosk | Facilities | Public | Low | Visitor sign-in | HQ lobby | 48h / n/a |

## How the copilot should use this
When the team names a host, account, or system, look it up here by name or alias and report its owner,
classification, criticality, and RTO/RPO. Let that raise or confirm severity (an incident on a Critical
or Restricted asset is at least SEV-1), decide who must be notified, and set the recovery priority. If
an asset is not listed, say so and ask the owning team; do not guess its criticality.


In [ ]:
%%writefile knowledge/isms_incident_management_policy.md
# SectraTEK - Incident Management Policy (SYNTHETIC, for training)

Part of the SectraTEK ISO/IEC 27001 ISMS. This policy sits above the operational IR plan and playbooks
and aligns with Annex A controls A.5.24-A.5.28 (incident management planning, assessment, response,
learning, and evidence). The IR plan and playbooks must stay consistent with this policy. Fictional,
for the Sectra First Responder lab.

## Purpose and scope
Ensure security incidents are reported, assessed, and responded to consistently, that evidence is
preserved, and that lessons improve the ISMS. Applies to all staff, systems, and third parties.

## Reporting and assessment (A.5.24 / A.5.25)
- All suspected incidents are reported to the SOC without delay via the incident channel.
- The SOC assesses and classifies each event using the IR plan's severity scale, informed by the
  classification and criticality of the assets involved (asset register).

## Response (A.5.26)
- Response follows the IR plan and the relevant playbook. Containment of production systems requires
  Incident Commander authorisation. Actions with external effect are approved and logged.

## Collection of evidence (A.5.28)
- Evidence is preserved before any destructive action, with recorded chain of custody, so it is
  admissible and the scope is defensible. Volatile data is captured first.

## Communication and notification
- Internal: management is notified for SEV-1 and SEV-2 incidents per the IR plan's matrix.
- External: customer, regulator, and partner notification is decided by Legal, driven by data
  classification and legal deadlines. No external statement is made without Legal and CISO approval.

## Learning from incidents (A.5.27)
- Every SEV-1 and every drill produces an after-action review within the stated window. Findings update
  the risk register, the controls, and the playbooks.

## How the copilot should use this
Keep all guidance within this policy: report and classify using asset context, preserve evidence before
containment, gate external actions on approval, and involve Legal for notification when classification
requires it. Cite this policy when explaining why a step is required.


In [ ]:
%%writefile knowledge/isms_information_classification_policy.md
# SectraTEK - Information Classification Policy (SYNTHETIC, for training)

Part of the SectraTEK ISO/IEC 27001 ISMS (aligns with Annex A control A.5.12, Classification of
information). Defines how information is classified and handled, which drives how seriously an incident
is treated and what notification obligations apply. Fictional, for the Sectra First Responder lab.

## Classification levels
- Restricted: most sensitive. Customer PII, cardholder data, credentials, authentication systems. Loss
  causes severe harm and regulatory exposure. Any incident touching Restricted data is at least SEV-1.
- Confidential: internal-only business data (finance, HR, internal documents). Loss causes significant
  harm. Incidents are SEV-1 or SEV-2 depending on scope.
- Internal: routine internal information not meant for the public. Limited harm if exposed.
- Public: approved for public release. No confidentiality impact.

## Handling rules
- Restricted and Confidential data is encrypted at rest and in transit, access is least-privilege, and
  access is logged and reviewed.
- Data may only be stored on approved systems in the asset register. Moving Restricted data to an
  unapproved location is itself a reportable incident.

## Breach and notification implications
- A confirmed compromise of Restricted personal data triggers the breach-notification process (see the
  Incident Management Policy): Legal assesses regulator and customer notification against the legal
  deadline. The classification of the data involved, not the size of the incident, drives this.
- Confidential business data exposure is reported to management; customer/regulator notification is
  assessed with Legal only if personal or contractual data is involved.

## How the copilot should use this
When an incident involves an asset, use its classification (from the asset register) to set the minimum
severity and to flag whether the breach-notification process may be triggered. Recommend involving Legal
early whenever Restricted personal data may be affected.


In [ ]:
%%writefile knowledge/isms_risk_register.md
# SectraTEK - ISMS Risk Register (SYNTHETIC, for training)

Part of the SectraTEK ISO/IEC 27001 ISMS. Records the information-security risks the organisation has
assessed, their rating, and how they are being treated. The incident copilot uses it to tie an active
incident back to a known risk and its treatment. Fictional, for the Sectra First Responder lab.

## Rating scale
Risk rating = Likelihood x Impact, expressed as Critical / High / Medium / Low. Treatment options:
Mitigate, Transfer, Avoid, Accept.

## Risk register
| Risk ID | Risk | Affected assets | Likelihood | Impact | Rating | Treatment | Owner |
|---|---|---|---|---|---|---|---|
| R-01 | Ransomware encrypts endpoints or servers, halting operations | FIN-WS-04, FS-01, DC-01 | Medium | Critical | High | Mitigate: EDR, offline immutable backups (BKP-01), segmentation, user training | CISO |
| R-02 | Theft of customer PII (exfiltration) | DB-CUST-01, FS-01 | Medium | Critical | High | Mitigate: DLP, egress filtering, least privilege, monitoring | Data Platform |
| R-03 | Credential phishing leading to account takeover | MAIL-365, EXEC-LT-02 | High | High | High | Mitigate: MFA, mail filtering, awareness; Transfer: cyber insurance | IT Infrastructure |
| R-04 | Compromise of privileged/admin accounts | DC-01 | Low | Critical | High | Mitigate: PAM, tiered admin, MFA, logging | IT Infrastructure |
| R-05 | Payment-system compromise / card data exposure | PAY-GW-01 | Low | Critical | High | Mitigate: PCI DSS controls, network isolation | Finance Systems |
| R-06 | Backup failure or compromise during an incident | BKP-01 | Low | High | Medium | Mitigate: immutable, segmented, tested restores | IT Infrastructure |
| R-07 | Insider misuse of access to sensitive data | HR-APP-01, FS-01 | Low | High | Medium | Mitigate: least privilege, access reviews, logging | HR / IT |

## How the copilot should use this
When an incident matches a known risk, name the Risk ID and its treatment so the team can see whether a
planned control held or failed, and so the lesson can be fed back into the risk register at the
after-action review. A control that failed is a risk-treatment gap to raise, not just an incident note.


In [ ]:
%%writefile knowledge/isms_statement_of_applicability.md
# SectraTEK - Statement of Applicability, selected controls (SYNTHETIC, for training)

An extract of the SectraTEK ISO/IEC 27001:2022 Statement of Applicability (SoA), limited to the Annex A
controls most relevant to incident response. The copilot can map an incident to the controls that
should have prevented or detected it, and to those that govern the response. Fictional, for the lab.

## Selected Annex A controls (2022)
| Control | Name | Status | Relevance to incidents |
|---|---|---|---|
| A.5.7 | Threat intelligence | Implemented | Feeds the watch-list used to recognise known-bad indicators |
| A.5.24 | Information security incident management planning and preparation | Implemented | The IR plan and playbooks exist and are drilled |
| A.5.25 | Assessment and decision on information security events | Implemented | Severity classification using asset context |
| A.5.26 | Response to information security incidents | Implemented | Playbook-driven, approval-gated response |
| A.5.27 | Learning from information security incidents | Implemented | After-action review updates the ISMS |
| A.5.28 | Collection of evidence | Implemented | Chain-of-custody evidence handling |
| A.8.7 | Protection against malware | Implemented | EDR / anti-malware on endpoints and servers |
| A.8.13 | Information backup | Implemented | Offline, immutable, tested backups (BKP-01) |
| A.8.15 | Logging | Implemented | Security logs retained for detection and investigation |
| A.8.16 | Monitoring activities | Implemented | Alerting on anomalous activity |
| A.5.14 | Information transfer | Implemented | Controls on data egress, relevant to exfiltration |

## How the copilot should use this
When useful, name the control an incident relates to: the control that failed (a prevention/detection
gap to raise at the after-action review) or the control governing the response (for example A.5.28 for
evidence). Keep this to a brief, relevant reference, not a full audit.


In [ ]:
%%writefile knowledge/playbook_data_exfiltration.md
# Data Exfiltration Playbook (SYNTHETIC, for training)

SectraTEK SOC playbook for suspected or confirmed theft of customer or sensitive data, including
DNS-tunnelling and cloud-upload exfiltration. Use with the IR Plan. Fictional, for the Sectra drill.

## Classification
Suspected or confirmed theft of customer or sensitive data is SEV-1. Notify management and Legal
immediately: data-breach notification deadlines may start the moment SectraTEK becomes aware. Keep it
SEV-1 until the scope of data moved is bounded by evidence.

## Indicators
- Large or unusual outbound transfers to an unfamiliar host; a spike in DNS queries to a single rare
  domain or many random subdomains of it (a sign of DNS tunnelling).
- Cloud-storage uploads outside normal business use; access to sensitive shares by an account that does
  not normally touch them; archiving tools (7-Zip, rar) running just before an upload.
- Activity outside the user's normal hours, or a burst of reads across many sensitive files.

## Immediate response
1. Declare SEV-1 and engage Legal early (breach obligations may apply with hard deadlines).
2. Identify the account and host involved; prepare to disable the account and isolate the host.
3. PRESERVE network and host evidence BEFORE blocking, so the volume and nature of data moved can be
   measured - blocking first can destroy the record of what left.
4. Then contain: isolate the host, disable the account, and block the destination domain/IP.
5. Notify management and Legal using the template below.

## Evidence collection
- Firewall, proxy, and DNS logs for the host across the event window: destinations, bytes out, and
  query volume per domain (to quantify what was tunnelled or uploaded).
- NetFlow or packet capture if available, to size the transfer.
- Host artifacts: the archiving-tool execution, the staged archive, command history, and any uploader
  binary or script (preserve and hash each).
- Access logs for the sensitive shares or database: which records the account touched and when.
Measuring scope is the goal: Legal needs an evidenced count of what data left, not a guess.

## Containment
- Block the exfiltration destination (domain and IP) at the firewall/DNS; isolate the involved host.
- Disable the account and rotate its credentials; revoke tokens and keys it held.
- If a service or API key was used, rotate it and review what else it could reach.
IT on-call runs these after the IC approves; the copilot may only simulate them.

## Eradication
- Remove the staging archive, the uploader/tunnelling tool, and any persistence that enabled access.
- Close the access path: revoke the over-broad permissions, reset the account, and fix the control that
  let a non-owner reach the sensitive data.
- Confirm no secondary channels (a second account, a scheduled job) remain.

## Recovery
- Restore normal service once the path is closed and credentials rotated; monitor the destination
  indicators, the account, and the sensitive shares for renewed activity for at least two weeks.
- Give Legal the final evidenced scope so any required regulator/customer notification can be completed
  within the deadline.
- Close only when the IC confirms the channel is shut and scope is documented.

## Management notification template
To: management@sectratek.example
Subject: SEV-1 Security Incident - Suspected data exfiltration
Team, the SOC is responding to suspected exfiltration of [data type] from [system/host]. We have
preserved network and host evidence and are measuring the volume and nature of data that left before
fully blocking the channel. Legal is engaged to assess notification obligations. Current scope: [one-line
factual status]. We will update within 60 minutes or on any material change. Incident Commander: [name].
Reference: [ticket].

## MITRE ATT&CK reference (for analysts)
Common techniques: Exfiltration Over C2 Channel (T1041), Exfiltration Over Alternative Protocol / DNS
(T1048), Archive Collected Data (T1560), Automated Exfiltration (T1020), Valid Accounts (T1078).

## Decision points
- Is the data personal or regulated? This drives Legal's notification clock.
- Can the volume moved be measured from logs? If not, preserve more before blocking.
- Was the access authorised but misused (insider) or external? This changes containment and HR/Legal.

## After-action
Record what data left, the channel, dwell time, and how it was detected. Feed improvements (egress
filtering, DLP, least-privilege on sensitive shares, DNS monitoring) back into Preparation and update
this playbook.


In [ ]:
%%writefile knowledge/playbook_phishing.md
# Phishing Playbook (SYNTHETIC, for training)

SectraTEK SOC playbook for reported phishing, credential harvesting, and malicious attachments. Use
with the IR Plan. Fictional, for the Sectra First Responder drill.

## Classification
A reported phishing email with no clicks and no entered credentials is SEV-3. If a user entered
credentials on a linked page, or opened/ran an attachment, treat it as SEV-2 and notify management.
If the compromised account is privileged, or many users are affected, escalate to SEV-1.

## Indicators
- A reported suspicious email; a mismatched sender and display name; a look-alike or newly registered
  sender domain; urgent or threatening language pushing the user to act fast.
- A user reports entering their password on a linked page, or opening an attachment that asked to enable
  content or macros.
- Mail-gateway alerts; an unusual sign-in location or new device for a user shortly after the email;
  a mailbox rule created to auto-forward or hide messages (a classic post-compromise move).

## Immediate response
1. Ask the reporting user NOT to delete the email; have them forward it as an attachment to
   soc@sectratek.example so headers are preserved.
2. If credentials were entered: reset that user's password and revoke all active sessions and tokens
   (IT on-call), then check for and remove any attacker-created mailbox forwarding rules.
3. Search the mail system for every recipient of the same message and quarantine/recall it.
4. If an attachment was opened or run, isolate that host and treat it as possible malware (pivot to the
   ransomware/malware response for evidence collection).
5. Notify management if this is SEV-2 or above, using the template below.

## Evidence collection
- The original email as an attachment, with full headers (sender IP, authentication results, path).
- The URLs and any attachment, detonated only in a safe sandbox, never on a corporate host.
- Sign-in and audit logs for the affected account: successful and failed logins, locations, devices,
  MFA prompts, and any new mailbox rules or OAuth app grants.
- If an attachment ran: host logs, process creation events, and any dropped files (preserve and hash).
Record custody for every item and work from copies.

## Containment
- Quarantine the phishing message across all mailboxes; block the sender domain and any payload URLs.
- Disable or force-reset the affected account; revoke sessions and tokens; remove malicious mailbox
  rules and unauthorised OAuth grants.
- If a host was involved, isolate it at the network until it is cleared.
IT on-call runs these with IC approval; the copilot may only simulate them.

## Eradication
- Remove the malicious mailbox rules, OAuth app consents, and any persistence on an affected host.
- Reset credentials for the affected user and any shared/service account they could reach.
- Close the delivery gap: tune the mail gateway, block the look-alike domain, and brief the user.

## Recovery
- Restore normal access once the account is secured and sessions are clean; monitor the account and the
  mailbox for re-compromise (new rules, odd sign-ins) for at least a week.
- Confirm no onward movement occurred from the account before closing.
- Record the lure and send a short, blameless awareness note to the affected team.

## Management notification template
To: management@sectratek.example
Subject: Security Incident - Phishing with credential compromise
Team, the SOC is responding to a phishing incident in which [N] user(s) [entered credentials / ran an
attachment]. Affected account(s) have been reset and sessions revoked; the message has been quarantined
across the organisation. Current scope: [one-line factual status]. We are reviewing sign-in logs for any
further access and will update within 60 minutes or on any material change. Incident Commander: [name].
Reference: [ticket].

## MITRE ATT&CK reference (for analysts)
Common techniques: Phishing (T1566), Valid Accounts (T1078), Steal Web Session Cookie (T1539), Email
Collection / mailbox rules (T1114), Account Manipulation (T1098). Use to guide the hunt.

## Decision points
- Were credentials actually entered, or only exposed? This sets SEV-2 vs SEV-3.
- Is the account privileged or widely shared? If yes, escalate and widen the hunt.
- Did an attachment execute? If yes, run the malware/ransomware evidence steps on the host in parallel.

## After-action
Capture the lure, the delivery gap it exploited, and detection timing. Feed improvements (gateway rules,
MFA coverage, user training) back into Preparation and update this playbook.


In [ ]:
%%writefile knowledge/playbook_ransomware.md
# Ransomware Playbook (SYNTHETIC, for training)

SectraTEK SOC playbook for suspected or confirmed ransomware on an endpoint or server. Use with the
IR Plan. Everything here is fictional and for the Sectra First Responder drill.

## Classification
Ransomware is a SEV-1 (Critical) incident. Notify management immediately (see the notification template
below) and engage an Incident Commander. If the encryptor is still running or has spread beyond one
host, keep it SEV-1 and widen the response; do not downgrade until spread is ruled out from evidence.

## Indicators
- Files renamed with an unusual or random extension; ransom notes dropped in many folders (a common
  filename seen before is README_RESTORE.txt).
- Mass file-modification alerts in a short window; volume shadow copies (VSS) deleted; backup jobs
  suddenly failing.
- Windows Security log 1102 (audit log cleared), bursts of 4688 process-creation events launching
  scripting or archive tools, or a service installed for persistence.
- An interactive logon from an internal host that had just produced many failed-logon (4625) events,
  suggesting a brute force that finally succeeded.

## Immediate response (first 15 minutes)
1. Declare SEV-1 and assign an Incident Commander. Open the incident bridge (out-of-band).
2. ISOLATE the affected host(s) from the network. Do NOT power off - memory is evidence and may hold
   keys or the encryptor's state. Isolation stops spread without destroying that evidence.
3. Disable the compromised user and service accounts (IT on-call, with IC approval).
4. Identify whether backups are intact and offline; protect them from the attacker immediately.
5. Notify management using the template in this playbook, and engage Legal (data theft often precedes
   ransomware, so a breach may also apply).

## Evidence collection
Preserve before eradicating. Collect, hashing each item and recording custody:
- Volatile first: a memory image of the affected host, and current network connections and process list.
- The ransom note file and a sample of encrypted files (for identification of the ransomware family).
- Windows Event Logs (Security, System, Application), especially IDs 1102, 4624, 4625, 4688, 7045.
- The affected user's command history and any dropped binaries from Downloads/Temp.
- Firewall and DNS logs around the event window, to find command-and-control or exfiltration traffic.
Work from read-only copies in the case folder; never analyse the only copy.

## Containment
- Short-term: keep the host isolated at the switch/firewall; block any identified C2 domains and IPs;
  disable the involved accounts and rotate their credentials.
- Long-term: segment the affected subnet, force a password reset for any accounts seen on the host, and
  hunt for the same indicators on other hosts before reconnecting anything.
Containment actions are run by IT on-call after the IC approves; the copilot may only simulate them.

## Eradication
- Remove persistence: the installed service, scheduled tasks, run keys, and any dropped loaders.
- Rebuild affected hosts from a known-good image rather than cleaning in place - ransomware rarely comes
  alone, and a cleaned host cannot be trusted.
- Reset all credentials used on the host, including any cached domain or service accounts.
- Confirm the initial access vector is closed (patched, reconfigured, or the account secured) so the
  attacker cannot simply return.

## Recovery
- Restore data only from backups taken before the compromise and verified clean; never pay without
  Legal/Exec direction and never restore onto an un-rebuilt host.
- Validate restored systems (integrity, function) in a monitored state before returning them to service.
- Monitor the environment closely for recurrence for at least two weeks: watch for the same indicators,
  new persistence, and repeat brute-force attempts.
- Close the incident only when the IC confirms eradication held and business function is restored.

## Management notification template
To: management@sectratek.example
Subject: SEV-1 Security Incident - Suspected ransomware
Team, the SOC is responding to a suspected ransomware incident affecting [host/business area]. The
affected system(s) have been isolated and the response team is following the IR plan. At this stage we
have [one-line factual status]. We are preserving evidence, assessing scope and backups, and have
engaged Legal. We will send the next update within 60 minutes or sooner on any material change.
Incident Commander: [name]. Reference: [ticket].

## MITRE ATT&CK reference (for analysts)
Common techniques: Valid Accounts (T1078), Brute Force (T1110), Command and Scripting Interpreter
(T1059), Inhibit System Recovery / delete shadow copies (T1490), Data Encrypted for Impact (T1486),
Indicator Removal / clear logs (T1070). Use these to guide the hunt, not as proof.

## Decision points
- Is spread confirmed contained? If not, stay SEV-1 and widen isolation.
- Are clean offline backups available? This decides the recovery path.
- Is there evidence of data theft before encryption? If yes, trigger the data-exfiltration playbook and
  breach assessment in parallel.

## After-action
Record the timeline, root cause, dwell time, what worked, and what to improve. Feed fixes (patching,
MFA, backup hardening, segmentation) back into Preparation and update this playbook.


In [ ]:
%%writefile knowledge/threat_intel.md
# SectraTEK - Threat Intelligence (SYNTHETIC, for training)

This is the company's small internal watch-list. The incident copilot compares indicators it extracts
from collected artifacts against this list (the `check_indicators` tool). Everything here is fictional
and exists only for the Sectra First Responder lab.

## Known-bad indicators
These have been seen in prior SectraTEK incidents or came from a trusted sharing group. Treat a match as
high-signal and tie it back to the relevant playbook. Format is one indicator per line: the value, then
two or more spaces, then a short note.

- exfil-sync.example          domain   Known data-exfiltration / DNS-tunnelling C2. Any subdomain counts.
- su.exe                  file     Dropper/loader used to install a persistence service.
- SysUpdateSvc            service  Fake "update" service used for persistence.
- 203.0.113.47          ip       C2 node seen in a prior exfiltration case.
- README_RESTORE.txt      file     Ransom note filename seen in a prior ransomware case.
- update-sync.example         domain   Look-alike callback domain from a prior phishing-to-malware case.
- 198.51.100.90            ip       Credential-phishing landing-page host seen before.

## How to use this list
The copilot extracts candidate indicators (domains, IPs, file names, services) from the artifacts the
team collects, then checks them against the values above. A match is a lead, not proof: confirm it in the
evidence and record it in the incident timeline. New confirmed indicators from an incident should be
added here during the after-action review so the next response is faster.


In [ ]:
%%writefile evidence/dns_queries.csv
timestamp,host,query,count
2024-07-28T09:30:00,10.4.2.19,www.microsoft.com,14
2024-07-28T09:31:00,10.4.2.19,outlook.office365.com,22
2024-07-28T09:35:00,10.4.2.19,sectratek.example,9
2024-07-28T09:40:00,10.4.2.19,a7f3k9d2.x1y2.exfil-sync.example,318
2024-07-28T09:41:00,10.4.2.19,b2c8m4e1.x1y2.exfil-sync.example,297
2024-07-28T09:42:00,10.4.2.19,update.googleapis.com,6
2024-07-28T09:45:00,10.4.2.19,c9d1n7f3.x1y2.exfil-sync.example,264


In [ ]:
%%writefile evidence/security_events.log
# SYNTHETIC Windows Security log excerpt (fictional, for training)
# fields: time  EventID  description
2024-07-28T09:41:12  4625  Failed logon  account=jmalik  src=10.4.2.19  reason=bad_password
2024-07-28T09:41:18  4625  Failed logon  account=jmalik  src=10.4.2.19  reason=bad_password
2024-07-28T09:41:25  4625  Failed logon  account=jmalik  src=10.4.2.19  reason=bad_password
2024-07-28T09:41:31  4625  Failed logon  account=administrator  src=10.4.2.19  reason=bad_password
2024-07-28T09:42:03  4624  Successful logon  account=jmalik  src=10.4.2.19  logon_type=3
2024-07-28T09:43:50  4688  Process created  user=jmalik  image=C:\Windows\System32\cmd.exe  cmdline="cmd.exe /c vssadmin delete shadows /all /quiet"
2024-07-28T09:44:02  4688  Process created  user=jmalik  image=C:\Users\jmalik\Downloads\7z.exe  cmdline="7z a -p backup.7z C:\Finance\*"
2024-07-28T09:45:10  7045  Service installed  name=SysUpdateSvc  image=C:\Windows\Temp\su.exe  start=auto
2024-07-28T09:47:33  1102  The audit log was cleared  by=jmalik
2024-07-28T09:48:00  4688  Process created  user=jmalik  image=C:\Windows\Temp\su.exe  cmdline="su.exe -run"


### 2. Connect the model and wire up Sectra

In [ ]:
import os, getpass
import irkit as ik

PROVIDER = 'groq'   # free key at console.groq.com  |  or 'openrouter' or 'ollama' (local Qwen)
if PROVIDER == 'groq' and not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()
if PROVIDER == 'openrouter' and not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Paste your OpenRouter API key (hidden): ').strip()
client, MODEL = ik.connect_model(PROVIDER)
print('Model:', MODEL)   # the first free model your key can reach

knowledge = ik.load_knowledge('knowledge')
ACTION_LOG = []
TOOLS = ik.make_ir_tools(knowledge, 'evidence', ACTION_LOG)
PROMPT = ik.build_ir_system_prompt(TOOLS)
conversation = []   # Sectra remembers the chat across talk() calls
print('Ready. Playbooks:', ', '.join(knowledge))

### 3. Talk to Sectra

`talk()` prints Sectra's tool steps and its reply, and remembers the conversation.

In [ ]:
def talk(message):
    print('You:', message, '\n')
    def show(ev):
        if 'final_answer' in ev:
            print('\nSectra:', ev['final_answer'])
            if ev.get('evidence_path'): print('   playbook:', ev['evidence_path'])
        else:
            print(f"  [{ev['step']}] {ev['action']}({ev['args']}) -> {ev['observation'][:70].strip()}")
    move = ik.run_agent(message, client, MODEL, TOOLS, PROMPT, on_step=show, history=conversation)
    conversation.append((message, move['final_answer'] if move else '(no answer)'))
    return move

_ = talk('We think we have ransomware on a finance workstation.')

### 4. Walk the incident

Run these in order. Watch Sectra: it tells you what the playbook says, **drafts** the notice and
waits for approval, **simulates** the send, **recommends** collection, then **runs** real analysis on
the bundled artifacts (`security_events.log`, `dns_queries.csv`) and finally writes the report.

In [ ]:
_ = talk('Draft the management notification.')

In [ ]:
_ = talk('Approve - send it.')

In [ ]:
_ = talk('What evidence should we collect, and how?')

In [ ]:
_ = talk('We uploaded security_events.log - build a timeline of what happened.')

In [ ]:
_ = talk('Pull the indicators out of dns_queries.csv and check them against the watch-list.')

In [ ]:
_ = talk('What should we do to contain and eradicate this?')

In [ ]:
_ = talk('Write the after-action report.')

### 5. Your turn

Try a different incident (*"a user clicked a phishing link"*, *"we see large uploads from one host"*),
drop your own `.md` playbooks into the `knowledge/` folder, or your own artifacts into `evidence/`.

`print('\n'.join(ACTION_LOG))`  shows everything Sectra did (all simulated).

In [ ]:
print('Action log (drill):')
print('\n'.join('- ' + e for e in ACTION_LOG) or '(nothing yet)')